### trade off between recall and precision
* 정밀도와 재현율은 상호 보완적인 평가 지표이기 때문에, 어느 한쪽을 강제로 높히면, 다른 한쪽은 떨어지기 쉽다. 이를 정밀도 - 재현율 트레이드 오프라고 부른다.
	- 일반적으로 분류의 결정 임계값(Threshold)를 조절하므로써, 두 지표 중 하나를 높힐 수 있다.
	- 여기서 임계값(Threshold)이란, Positive를 예측하는 임계값을 의미한다. 즉, 임계값이 클수록 Positive예측을 덜 하고, 
		임계값이 작을수록 Positive 예측을 많이 한다.

* 일반적으로 이진 분류에서는 임계값을 0.5, 즉 50%로 정하고 이 기준보다 높은 확률 값을 가지는 레이블을 예측 레이블로 선택한다.
	- 이를 Classifier Estimator에서는 `predict_proba()`라는 API를 이용해서 개별 레이블에 대한 확률 값을 확인할 수 있다.

* Recall과 Precision이 Threshold값에 의해 trade off 관계를 가지는 이유는, Threshold값에 의해서 Positive 레이블에 대한 예측을 조절하게 되고, 임계값이 작아지면 더 높은 확률로 Positive로 예측하므로, TP와 FP가 늘어나는 반면, FN은 줄어들 수 밖에 없다. 따라서, Recall의 성능 평가는 좋아지는 반면, Precision의 성능 평가는 낮아질 수 밖에 없다. 역의 관계도 성립한다.

### 결론
* 가장 중요한 것은 두 지표의 Balance이다. 하나의 지표가 100%라도, 다른 지표각 0%이면 아무런 효용도 갖지 못한다.

In [5]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder

# Null 처리 함수
def fillna(df):
	df['Age'] = df['Age'].fillna(df['Age'].mean())
	df['Cabin'] = df['Cabin'].fillna('N')
	df['Embarked'] = df['Embarked'].fillna('N')
	df['Fare'] = df['Fare'].fillna(0)
	return df

# 머신러닝 알고리즘에 불필요한 피처 제거
def drop_features(df):
	df.drop(['PassengerId', 'Name', 'Ticket'], axis=1, inplace=True)
	return df

# 레이블 인코딩 수행.
def format_features(df):
	df['Cabin'] = df['Cabin'].str[:1]
	features = ['Cabin', 'Sex', 'Embarked']
	for feature in features:
		le = LabelEncoder()
		le = le.fit(df[feature])
		df[feature] = le.transform(df[feature])
	return df

# 앞에서 설정한 데이터 전처리 함수 호출
def transform_features(df):
	df = fillna(df)
	df = drop_features(df)
	df = format_features(df)
	return df

In [6]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

titanic_df = pd.read_csv('titanic_train.csv')
y_titanic_df = titanic_df['Survived']
X_titanic_df = titanic_df.drop('Survived', axis=1)
X_titanic_df = transform_features(X_titanic_df)
X_titanic_df['Age'].fillna(X_titanic_df['Age'].mean(), inplace=True)
X_train, X_test, y_train, y_test = train_test_split(X_titanic_df, y_titanic_df, test_size=0.2, random_state=11)

lr_clf = LogisticRegression(solver='liblinear')
lr_clf.fit(X_train, y_train)

/var/folders/c1/h7bpvp6j477dywpxvwgcb5qw0000gn/T/ipykernel_40142/1641186859.py:9: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  X_titanic_df['Age'].fillna(X_titanic_df['Age'].mean(), inplace=True)


,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`multiclass` problems (`n_classes >= 3`), all solvers except 'liblinear' minimize the full multinomial loss, 'liblinear' will raise an error.- 'newton-cholesky' is a good choice for `n_samples` >> `n_features * n_classes`, especially with one-hot encoded categorical features with rare categories. Be aware that the memory usage of this solver has a quadratic dependency on `n_features * n_classes` because it explicitly computes the full Hessian matrix.- For small datasets, 'liblinear' is a good choice, whereas 'sag' and 'saga' are faster for large ones;- 'liblinear' can only handle binary classification by default. To apply a one-versus-rest scheme for the multiclass setting one can wrap it with the :class:`~sklearn.multiclass.OneVsRestClassifier`... warning:: The choice of the algorithm depends on the penalty chosen (`l1_ratio=0` for L2-penalty, `l1_ratio=1` for L1-penalty and `0 < l1_ratio < 1` for Elastic-Net) and on (multinomial) multiclass support: ================= ======================== ====================== solver l1_ratio multinomial multiclass ================= ======================== ====================== 'lbfgs' l1_ratio=0 yes 'liblinear' l1_ratio=1 or l1_ratio=0 no 'newton-cg' l1_ratio=0 yes 'newton-cholesky' l1_ratio=0 yes 'sag' l1_ratio=0 yes 'saga' 0<=l1_ratio<=1 yes ================= ======================== ======================.. note:: 'sag' and 'saga' fast convergence is only guaranteed on features with approximately the same scale. You can preprocess the data with a scaler from :mod:`sklearn.preprocessing`... seealso:: Refer to the :ref:`User Guide <Logistic_regression>` for more information regarding :class:`LogisticRegression` and more specifically the :ref:`Table <logistic_regression_solvers>` summarizing solver/penalty supports... versionadded:: 0.17 Stochastic Average Gradient (SAG) descent solver. Multinomial support in version 0.18... versionadded:: 0.19 SAGA solver... versionchanged:: 0.22 The default solver changed from 'liblinear' to 'lbfgs' in 0.22... versionadded:: 1.2 newton-cholesky solver. Multinomial support in version 1.6.",'liblinear'
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some pen

In [9]:
pred_proba = lr_clf.predict_proba(X_test)

print(pred_proba[:3])

[[0.44935225 0.55064775]
 [0.86335511 0.13664489]
 [0.86429643 0.13570357]]


In [19]:
from sklearn.preprocessing import Binarizer
import numpy as np

custom_binarizer = 0.4

binarizer = Binarizer(threshold=custom_binarizer).fit(pred_proba[:, 1].reshape(-1, 1))
custom_prediction = binarizer.transform(pred_proba[:, 1].reshape(-1, 1))

print(np.concatenate([pred_proba[:, 1].reshape(-1, 1), custom_prediction.reshape(-1, 1)], axis=1))

[[0.55064775 1.        ]
 [0.13664489 0.        ]
 [0.13570357 0.        ]
 [0.15031481 0.        ]
 [0.1765659  0.        ]
 [0.15768776 0.        ]
 [0.12904511 0.        ]
 [0.72771397 1.        ]
 [0.21814872 0.        ]
 [0.66814003 1.        ]
 [0.13821237 0.        ]
 [0.12941902 0.        ]
 [0.13574049 0.        ]
 [0.12934056 0.        ]
 [0.43966456 1.        ]
 [0.14996977 0.        ]
 [0.11045828 0.        ]
 [0.25749268 0.        ]
 [0.28879776 0.        ]
 [0.76223723 1.        ]
 [0.24315894 0.        ]
 [0.37571831 0.        ]
 [0.15344754 0.        ]
 [0.17288743 0.        ]
 [0.13174371 0.        ]
 [0.22996172 0.        ]
 [0.17053651 0.        ]
 [0.09663869 0.        ]
 [0.26627952 0.        ]
 [0.31152613 0.        ]
 [0.92353131 1.        ]
 [0.7746788  1.        ]
 [0.12838061 0.        ]
 [0.75924583 1.        ]
 [0.37288269 0.        ]
 [0.22996172 0.        ]
 [0.09445724 0.        ]
 [0.59397426 1.        ]
 [0.06956415 0.        ]
 [0.1234948  0.        ]


In [21]:
from sklearn.metrics import precision_recall_curve
import numpy as np

predict_proba_1 = lr_clf.predict_proba(X_test)[:, 1]
precisions, recalls, thresholds = precision_recall_curve(y_test, predict_proba_1)

index = np.arange(0, thresholds.shape[0], 15)

print(np.round(thresholds[index], 2))
print(np.round(precisions[index], 3))
print(np.round(recalls[index], 3))

[0.02 0.11 0.13 0.14 0.16 0.24 0.32 0.45 0.62 0.73 0.87]
[0.341 0.372 0.401 0.44  0.505 0.598 0.688 0.774 0.915 0.968 0.938]
[1.    1.    0.967 0.902 0.902 0.902 0.869 0.787 0.705 0.492 0.246]
